# ROS06 · Executors, callback groups and lifecycle nodes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS06_executors_callback_groups_and_lifecycle_nodes.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 60 min

**Skill Hub concepts:** Executors and callback groups; Lifecycle (managed) nodes

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS06")

## Map: who runs your callbacks, and when

| Executor | Threads | Effect |
|---|---|---|
| `SingleThreadedExecutor` (what `rclpy.spin` uses) | 1 | callbacks run one at a time; a slow one delays all others |
| `MultiThreadedExecutor` | N | callbacks may run in parallel, **subject to callback groups** |

| Callback group | Rule inside the group |
|---|---|
| `MutuallyExclusiveCallbackGroup` (default for every node) | never two callbacks of the group at once |
| `ReentrantCallbackGroup` | any number at once, even the same callback twice |

Classic deadlock: in a single-threaded executor, a callback calls a service and blocks waiting for the reply, but the
reply can only be processed by the same (busy) thread.

**Lifecycle (managed) nodes:** a standard state machine so a launch system or supervisor brings drivers up in order.

```
 Unconfigured ──configure──> Inactive ──activate──> Active
      ^  <──cleanup──────────┘   ^   <──deactivate──┘
      └──────── any state ── shutdown ──> Finalized          (primary states: 4)
```
Lifecycle publishers only send while the node is Active.

`ros2 lifecycle get /node`, `ros2 lifecycle set /node configure|activate|deactivate|cleanup|shutdown`

In [ ]:
from roboprep import ros

## Task 1: keep a service responsive next to a slow timer
The timer simulates 0.5 s of work every second. The `/busy_node/ping` service must answer within 0.35 s at any time (a blocked executor takes up to 0.5 s).
Fix the executor and callback groups (do not shorten the work).

In [ ]:
%%writefile busy_node.py
import time
import rclpy
from rclpy.callback_groups import MutuallyExclusiveCallbackGroup
from rclpy.executors import ExternalShutdownException, MultiThreadedExecutor
from rclpy.node import Node
from std_srvs.srv import Trigger


class Busy(Node):
    def __init__(self):
        super().__init__("busy_node")
        self.create_timer(1.0, self.work)                         # TODO
        self.create_service(Trigger, "/busy_node/ping", self.ping)

    def work(self):
        time.sleep(0.5)      # pretend: image processing, a blocking driver call, ...

    def ping(self, req, resp):
        resp.success, resp.message = True, "pong"
        return resp


def main():
    rclpy.init()
    node = Busy()
    try:
        rclpy.spin(node)          # TODO
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
lab.check("executor")

In [ ]:
%%writefile busy_node.cpp
// C++ (rclcpp) version of Task 1 (optional, same checker). Grade it with lab.check("executor", "busy_node.cpp"):
// the grader builds it with colcon as package rdl_busy_node, executable busy_node (depends on rclcpp, std_srvs).
#include <chrono>
#include <thread>
#include <rclcpp/rclcpp.hpp>
#include <std_srvs/srv/trigger.hpp>
using namespace std::chrono_literals;

class Busy : public rclcpp::Node {
public:
    Busy() : Node("busy_node") {
        timer_ = create_wall_timer(1s, [] { std::this_thread::sleep_for(500ms); });   // TODO: callback group
        srv_ = create_service<std_srvs::srv::Trigger>("/busy_node/ping",
            [](const std::shared_ptr<std_srvs::srv::Trigger::Request>, std::shared_ptr<std_srvs::srv::Trigger::Response> r) {
                r->success = true;
                r->message = "pong";
            });                                                                          // TODO: callback group
    }
private:
    rclcpp::TimerBase::SharedPtr timer_;
    rclcpp::Service<std_srvs::srv::Trigger>::SharedPtr srv_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<Busy>());   // TODO: which executor lets ping answer while work sleeps?
    rclcpp::shutdown();
}

In [ ]:
# lab.check("executor", "busy_node.cpp")  # uncomment after writing the C++ cell above

## Task 2: a managed sensor node
Node `lc_sensor`. `on_configure`: create a lifecycle publisher (`std_msgs/Float32` on `/lc_sensor/data`) and a 10 Hz
timer that publishes a counter. `on_activate` / `on_deactivate`: let the base class switch the publisher.
`on_cleanup`: destroy the timer and the publisher.

In [ ]:
%%writefile lc_sensor.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.lifecycle import LifecycleNode, TransitionCallbackReturn
from std_msgs.msg import Float32


class LcSensor(LifecycleNode):
    def __init__(self):
        super().__init__("lc_sensor")
        self.pub, self.timer, self.n = None, None, 0

    def on_configure(self, state):
        return TransitionCallbackReturn.SUCCESS   # TODO

    def tick(self):
        self.n += 1
        if self.pub is not None:
            self.pub.publish(Float32(data=float(self.n)))

    def on_activate(self, state):
        return super().on_activate(state)       # activates the lifecycle publishers

    def on_deactivate(self, state):
        return super().on_deactivate(state)

    def on_cleanup(self, state):
        return TransitionCallbackReturn.SUCCESS   # TODO


def main():
    rclpy.init()
    try:
        rclpy.spin(LcSensor())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
import time
ros.start("python lc_sensor.py", log="lc.log")
time.sleep(2)
ros.ros("ros2 lifecycle get /lc_sensor; ros2 lifecycle set /lc_sensor configure; ros2 lifecycle set /lc_sensor activate; "
        "ros2 lifecycle get /lc_sensor; timeout 3 ros2 topic hz /lc_sensor/data", timeout=30)
ros.stop_all()

In [ ]:
lab.check("lifecycle")

In [ ]:
%%writefile lc_sensor.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("lifecycle", "lc_sensor.cpp"):
// the grader builds it with colcon as package rdl_lc_sensor, executable lc_sensor (depends on rclcpp, rclcpp_lifecycle, std_msgs).
#include <chrono>
#include <rclcpp/rclcpp.hpp>
#include <rclcpp_lifecycle/lifecycle_node.hpp>
#include <std_msgs/msg/float32.hpp>
using namespace std::chrono_literals;
using CallbackReturn = rclcpp_lifecycle::node_interfaces::LifecycleNodeInterface::CallbackReturn;

class LcSensor : public rclcpp_lifecycle::LifecycleNode {
public:
    LcSensor() : LifecycleNode("lc_sensor") {}

    CallbackReturn on_configure(const rclcpp_lifecycle::State &) override {
        return CallbackReturn::SUCCESS;   // TODO: lifecycle publisher on /lc_sensor/data and a 10 Hz timer
    }
    CallbackReturn on_activate(const rclcpp_lifecycle::State &s) override {
        LifecycleNode::on_activate(s);
        return CallbackReturn::SUCCESS;
    }
    CallbackReturn on_deactivate(const rclcpp_lifecycle::State &s) override {
        LifecycleNode::on_deactivate(s);
        return CallbackReturn::SUCCESS;
    }
    CallbackReturn on_cleanup(const rclcpp_lifecycle::State &) override {
        return CallbackReturn::SUCCESS;   // TODO: release the timer and the publisher
    }
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    auto node = std::make_shared<LcSensor>();
    rclcpp::spin(node->get_node_base_interface());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("lifecycle", "lc_sensor.cpp")  # uncomment after writing the C++ cell above

## Checkpoint: Math
**M1 (Analyze).** Single-threaded executor; the timer callback blocks for 500 ms once per second; ping requests arrive
at random times. Worst-case ping latency in milliseconds (ignore transport)?

**M2 (Remember).** Number of primary lifecycle states?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Analyze).** Inside a subscription callback you call `client.call(request)` (synchronous) on a node spun by
`rclpy.spin`. What happens?
- a) it works but is slow
- b) deadlock: the response must be processed by the executor thread that is blocked inside your callback
- c) the request is dropped

**K2 (Evaluate).** Why do drivers (cameras, motor controllers) use lifecycle nodes? Describe a bring-up and an error
recovery sequence that a plain node cannot express.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros06.h`).
3. In `roboprep/checks/ros06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()